In [1]:
# # Create the Boltz cache directory
# !mkdir -p ./.boltz

# # Download the required files from the mirror
# !wget https://hf-mirror.com/boltz-community/boltz-2/resolve/main/mols.tar -P ./.boltz/
# !wget https://hf-mirror.com/boltz-community/boltz-2/resolve/main/boltz2_conf.ckpt -P ./.boltz/
# !wget https://hf-mirror.com/boltz-community/boltz-2/resolve/main/boltz2_aff.ckpt -P ./.boltz/

# # Extract the molecular data archive inside the cache directory
# !cd ./.boltz && tar xf mols.tar

In [2]:
import os
from kaggle_secrets import UserSecretsClient

# Initialize the client (note the 's' in Secrets)
user_secrets = UserSecretsClient()

# Retrieve and set the API key
os.environ["NVIDIA_API_KEY"] = user_secrets.get_secret("NVIDIA_API_KEY")

## Env.

In [3]:
# ============================================================
# Kaggle Environment Setup
# ============================================================
# Install dependencies
!pip install -q rdkit==2024.9.4 numpy==1.26.4
# Install Boltz with CUDA kernels enabled
!pip install -q "boltz-community[cuda]"

import torch
import subprocess
from pathlib import Path

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Make sure 'Accelerator' is set to GPU in the Kaggle notebook settings.")

# Verify the Boltz CLI is installed
!boltz --help > /dev/null && echo "Boltz CLI ready."

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opencv-python-headless 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
mne 1.13.2 requires numpy<3,>=2.1, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.14.0.94 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
bayesian-optimization 3.3.0 requires numpy>=2.1.3; python_full_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.

In [4]:
import numpy as np
import rdkit
import torch
import boltz

print("NumPy:", np.__version__)          # Likely 1.26.4
print("RDKit:", rdkit.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Boltz imported successfully")

NumPy: 2.2.6
RDKit: 2024.09.4
PyTorch: 2.11.0+cu128
CUDA available: True
Boltz imported successfully


## Chemistry

In [5]:
%%writefile chemistry.py
"""Local reaction generation and property checks for a frozen molecular library.

The library contains only molecules.sqlite and library.json. Product indices
refer to distinct canonical products in reaction-enumeration order, not sorted
SMILES. These checks do not establish historical novelty or set diversity.
"""

import hashlib
import json
import math
import re
import sqlite3
from collections import Counter
from functools import cache
from pathlib import Path

import rdkit
from rdkit import Chem
from rdkit.Chem import (
    AllChem,
    Crippen,
    Descriptors,
    GraphDescriptors,
    Lipinski,
    rdMolDescriptors,
)

RDKIT_VERSION = "2024.09.4"
PRODUCT_ID = re.compile(
    r"rxn:([0-9]+):([0-9]+):([0-9]+)(?::([0-9]+))?(?::k([0-9]+))?\Z"
)
FILTER_LIMITS = {
    "max_kappa1": "kappa1",
    "max_molecular_weight": "molecular_weight",
    "max_heavy_atoms": "heavy_atoms",
    "min_heavy_atoms": "heavy_atoms",
    "min_molecular_weight": "molecular_weight",
    "max_aromatic_rings": "aromatic_rings",
    "max_aliphatic_rings": "aliphatic_rings",
    "min_rings": "rings",
    "max_fsp3": "fsp3",
    "max_spiro_atoms": "spiro_atoms",
    "max_ring_size": "largest_ring",
    "max_aromatic_nitrogen": "aromatic_nitrogen",
    "max_aromatic_fraction": "aromatic_fraction",
    "max_aromatic_carbon": "aromatic_carbon",
    "max_halogens": "halogens",
    "max_nitrogen": "N",
    "max_oxygen": "O",
    "max_sulfur": "S",
    "max_phosphorus": "P",
    "max_iodine": "I",
    "max_bromine": "Br",
    "max_fluorine": "F",
    "min_heteroatom_fraction": "heteroatom_fraction",
    "max_heteroatom_fraction": "heteroatom_fraction",
    "max_hbd": "hbd",
    "max_amide_bonds": "amide_bonds",
    "max_nitrile": "nitrile",
    "max_hydroxyl": "hydroxyl",
    "max_triple_bonds": "triple_bonds",
    "min_rotatable_bonds": "rotatable_bonds",
    "max_rotatable_bonds": "rotatable_bonds",
    "max_bertz_ct": "bertz_ct",
    "max_hba": "hba",
    "max_kappa2": "kappa2",
    "max_kappa3": "kappa3",
    "max_stereocenters": "stereocenters",
    "max_logp": "logp",
}
EXCLUDED_PROPERTIES = ("metals", "B", "Si", "Se", "azide", "isocyanide", "allene")
GROUP_PATTERNS = {
    name: Chem.MolFromSmarts(smarts)
    for name, smarts in {
        "nitrile": "[CX2]#[NX1]",
        "azide": "[NX1]~[NX2]~[NX2,NX3]",
        "isocyanide": "[CX1-]#[NX2+]",
        "allene": "[CX3]=[CX2]=[CX3]",
        "hydroxyl": "[OX2H]",
    }.items()
}
DESCRIPTORS = {
    "molecular_weight": Descriptors.MolWt,
    "rings": Descriptors.RingCount,
    "aromatic_rings": Descriptors.NumAromaticRings,
    "aliphatic_rings": Descriptors.NumAliphaticRings,
    "fsp3": rdMolDescriptors.CalcFractionCSP3,
    "spiro_atoms": rdMolDescriptors.CalcNumSpiroAtoms,
    "amide_bonds": rdMolDescriptors.CalcNumAmideBonds,
    "hbd": Lipinski.NumHDonors,
    "hba": Lipinski.NumHAcceptors,
    "rotatable_bonds": Descriptors.NumRotatableBonds,
    "kappa1": GraphDescriptors.Kappa1,
    "kappa2": GraphDescriptors.Kappa2,
    "kappa3": GraphDescriptors.Kappa3,
    "bertz_ct": GraphDescriptors.BertzCT,
    "logp": Crippen.MolLogP,
}


def file_sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def validate_filters(limits):
    if not isinstance(limits, dict) or set(FILTER_LIMITS) - limits.keys():
        raise ValueError("Library is missing required property limits")
    if any(
        type(limits[k]) not in (int, float) or not math.isfinite(limits[k])
        for k in FILTER_LIMITS
    ):
        raise ValueError("Property limits must be finite numbers")


def property_reader(mol):
    """Return a lazy, cached reader for this molecule's published properties."""
    counts = Counter(atom.GetSymbol() for atom in mol.GetAtoms())
    aromatic = Counter(
        atom.GetSymbol() for atom in mol.GetAtoms() if atom.GetIsAromatic()
    )
    heavy = mol.GetNumHeavyAtoms()
    basic = dict(counts)
    basic.update(
        heavy_atoms=heavy,
        aromatic_nitrogen=aromatic["N"],
        aromatic_carbon=aromatic["C"],
        aromatic_fraction=sum(aromatic.values()) / max(heavy, 1),
        heteroatom_fraction=(heavy - counts["C"]) / max(heavy, 1),
        halogens=sum(counts[element] for element in ("F", "Cl", "Br", "I")),
        metals=sum(
            counts[element]
            for element in "Fe Zn Cu Mg Ca Mn Mo Ni Co Pt Au Hg Pd Ru W V Cr Cd Pb As Sb Bi Sn Al Li Na K".split()
        ),
    )

    @cache
    def read(name):
        if name in DESCRIPTORS:
            return DESCRIPTORS[name](mol)
        if name in GROUP_PATTERNS:
            return len(mol.GetSubstructMatches(GROUP_PATTERNS[name]))
        if name == "largest_ring":
            return max(map(len, mol.GetRingInfo().AtomRings()), default=0)
        if name == "triple_bonds":
            return sum(
                bond.GetBondType() == Chem.BondType.TRIPLE for bond in mol.GetBonds()
            )
        if name == "stereocenters":
            return len(Chem.FindMolChiralCenters(mol, includeUnassigned=True))
        return basic.get(name, 0)

    return read


def passes_filters(mol, limits):
    validate_filters(limits)
    value = property_reader(mol)
    for key, name in FILTER_LIMITS.items():
        actual = value(name)
        if not math.isfinite(actual):
            return False
        if (key.startswith("min_") and actual < limits[key]) or (
            key.startswith("max_") and actual > limits[key]
        ):
            return False
    return all(value(name) == 0 for name in EXCLUDED_PROPERTIES)


def role_mask(value):
    return (
        int.from_bytes(value, "big")
        if isinstance(value, (bytes, bytearray, memoryview))
        else int(value or 0)
    )


def smarts_products(left, right, smarts):
    """Keep the first product of each outcome; canonicalize and retain order."""
    try:
        reaction = AllChem.ReactionFromSmarts(smarts)
        reactants = (Chem.MolFromSmiles(left), Chem.MolFromSmiles(right))
        if reaction is None or any(mol is None for mol in reactants):
            return []
        products = {}
        for outcome in reaction.RunReactants(reactants):
            if outcome:
                try:
                    Chem.SanitizeMol(outcome[0])
                    products.setdefault(Chem.MolToSmiles(outcome[0]), None)
                except (ValueError, RuntimeError):
                    continue
        return list(products)
    except (ValueError, RuntimeError):
        return []


def join_synthons(left, right):
    """Join the neighbors of [1*] and [2*], removing the labelled dummy atoms."""
    try:
        molecules = (Chem.MolFromSmiles(left), Chem.MolFromSmiles(right))
        if any(mol is None for mol in molecules):
            return None
        sites = [
            next(
                (
                    atom
                    for atom in mol.GetAtoms()
                    if atom.GetAtomicNum() == 0 and atom.GetIsotope() == label
                ),
                None,
            )
            for label, mol in enumerate(molecules, 1)
        ]
        if any(atom is None or not atom.GetNeighbors() for atom in sites):
            return None
        first = Chem.RWMol(molecules[0])
        first_site = sites[0].GetIdx()
        first_neighbor = sites[0].GetNeighbors()[0].GetIdx()
        first.RemoveAtom(first_site)
        first_neighbor -= first_neighbor > first_site

        # Build the second fragment from atoms and bond types. This precisely
        # defines the synthon format's bond semantics, including product order.
        second = Chem.RWMol()
        mapping = {
            atom.GetIdx(): second.AddAtom(atom)
            for atom in molecules[1].GetAtoms()
            if atom.GetIdx() != sites[1].GetIdx()
        }
        for bond in molecules[1].GetBonds():
            begin, end = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            if begin in mapping and end in mapping:
                second.AddBond(mapping[begin], mapping[end], bond.GetBondType())
        neighbor = mapping[sites[1].GetNeighbors()[0].GetIdx()] + first.GetNumAtoms()
        product = Chem.RWMol(Chem.CombineMols(first, second))
        product.AddBond(first_neighbor, neighbor, Chem.BondType.SINGLE)
        Chem.SanitizeMol(product)
        return Chem.MolToSmiles(product)
    except (ValueError, RuntimeError):
        return None


class Library:
    def __init__(self, directory):
        root = Path(directory).resolve()
        self.manifest = json.loads((root / "library.json").read_text())
        manifest = self.manifest
        if (
            not isinstance(manifest, dict)
            or type(manifest.get("format_version")) is not int
            or manifest["format_version"] != 1
        ):
            raise ValueError("Unsupported library format")
        if (
            manifest.get("rdkit_version") != RDKIT_VERSION
            or rdkit.__version__ != RDKIT_VERSION
        ):
            raise ValueError(f"Use RDKit {RDKIT_VERSION} with this library")
        if not isinstance(manifest.get("allowed_reaction"), str) or not re.fullmatch(
            r"rxn:[0-9]+", manifest["allowed_reaction"]
        ):
            raise ValueError("Invalid library reaction")
        for field in ("case_digest", "database_sha256"):
            if not isinstance(manifest.get(field), str) or not re.fullmatch(
                r"[0-9a-f]{64}", manifest[field]
            ):
                raise ValueError(f"Invalid library {field}")
        validate_filters(manifest.get("filters"))
        self.database = root / "molecules.sqlite"
        if file_sha256(self.database) != manifest["database_sha256"]:
            raise ValueError("Library database hash does not match its manifest")
        self.reaction_id = int(manifest["allowed_reaction"].split(":")[1])
        # The snapshot hash covers this file only. Ignore adjacent WAL/journal
        # files so they cannot silently replace the hashed library contents.
        self.connection = sqlite3.connect(
            self.database.as_uri() + "?mode=ro&immutable=1", uri=True
        )
        try:
            self.connection.execute("PRAGMA query_only=ON")
            self.reaction()
            self.connection.execute(
                "SELECT mol_id,smiles,role_mask FROM molecules LIMIT 0"
            )
        except BaseException:
            self.connection.close()
            raise

    def close(self):
        self.connection.close()

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        self.close()

    def reaction(self):
        row = self.connection.execute(
            "SELECT smarts,roleA,roleB,roleC FROM reactions WHERE rxn_id=?",
            (str(self.reaction_id),),
        ).fetchone()
        if row is None:
            raise ValueError("Allowed reaction is absent from the library")
        return row[0], *(role_mask(value) for value in row[1:])

    def ids_for_role(self, required):
        if required <= 0:
            return []
        indexed = self.connection.execute(
            "SELECT 1 FROM sqlite_schema WHERE type='table' AND name='molecule_roles'"
        ).fetchone()
        if indexed:
            bits = [bit for bit in range(required.bit_length()) if required >> bit & 1]
            query = " INTERSECT ".join(
                "SELECT mol_id FROM molecule_roles WHERE bit=?" for _ in bits
            )
            return [
                row[0]
                for row in self.connection.execute(
                    f"SELECT m.mol_id FROM molecules m JOIN ({query}) r ON r.mol_id=m.mol_id",
                    bits,
                )
            ]
        return [
            identifier
            for identifier, mask in self.connection.execute(
                "SELECT mol_id,role_mask FROM molecules"
            )
            if role_mask(mask) & required == required
        ]

    def products(self, molecule_ids):
        if len(molecule_ids) not in (2, 3):
            return []
        rows = [
            self.connection.execute(
                "SELECT smiles,role_mask FROM molecules WHERE mol_id=?", (identifier,)
            ).fetchone()
            for identifier in molecule_ids
        ]
        if any(row is None for row in rows):
            return []
        smarts, role_a, role_b, role_c = self.reaction()
        masks = [role_mask(row[1]) for row in rows]
        has_role = lambda mask, required: required != 0 and mask & required == required
        if has_role(masks[0], role_a) and has_role(masks[1], role_b):
            first, second = rows[0][0], rows[1][0]
        elif has_role(masks[1], role_a) and has_role(masks[0], role_b):
            first, second = rows[1][0], rows[0][0]
        else:
            return []
        if len(rows) == 2:
            if self.reaction_id == 1:
                joined = join_synthons(first, second)
                return [joined] if joined else []
            return smarts_products(first, second, smarts)
        if not has_role(masks[2], role_c):
            return []
        third = rows[2][0]
        if self.reaction_id == 3:
            intermediate = join_synthons(first, second)
            return (
                smarts_products(
                    intermediate, third, "[C:1](=O)[OH].[N:2]>>[C:1](=O)[N:2]"
                )
                if intermediate
                else []
            )
        if self.reaction_id == 5:
            intermediates = smarts_products(
                first, second, "[#6:1][Br].[#6:2][B]([OH])[OH]>>[#6:1][#6:2]"
            )
            return (
                smarts_products(
                    intermediates[0],
                    third,
                    "[#6:1][Cl].[#6:2][B]([OH])[OH]>>[#6:1][#6:2]",
                )
                if intermediates
                else []
            )
        return []

    def resolve(self, product_id):
        match = PRODUCT_ID.fullmatch(product_id)
        if match is None or int(match[1]) != self.reaction_id:
            return None
        molecules = [int(value) for value in match.group(2, 3, 4) if value is not None]
        variants = self.products(molecules)
        index = int(match[5] or 0)
        return variants[index] if index < len(variants) else None


Overwriting chemistry.py



## Local Evaluation

In [6]:
%%writefile local_eval.py
"""Local validate/final for offline NIM runs.

Mirrors the owner's set-level semantics using only the frozen library.
NOT the owner's service.
"""

import math
from rdkit import Chem, DataStructs, RDLogger
RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem

from chemistry import passes_filters


def _fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return AllChem.GetMorganGenerator(radius=2, fpSize=2048).GetFingerprint(mol) if mol else None


def validate_set(product_ids, library, max_tanimoto=0.8):
    reasons = []
    smiles = {}
    for pid in product_ids:
        smi = library.resolve(pid)
        if smi is None:
            reasons.append(f"{pid}: unresolved"); continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None or not passes_filters(mol, library.manifest["filters"]):
            reasons.append(f"{pid}: filter failure"); continue
        smiles[pid] = smi
    fps = {pid: _fp(s) for pid, s in smiles.items()}
    fps = {p: f for p, f in fps.items() if f is not None}
    pids = list(fps)
    for i in range(len(pids)):
        for j in range(i + 1, len(pids)):
            sim = DataStructs.TanimotoSimilarity(fps[pids[i]], fps[pids[j]])
            if sim > max_tanimoto:
                reasons.append(f"{pids[i]}~{pids[j]}: Tanimoto {sim:.2f} > {max_tanimoto}")
    return (not reasons, reasons, smiles)


def final_set(product_ids, library, oracle, max_tanimoto=0.8):
    ok, reasons, smiles = validate_set(product_ids, library, max_tanimoto)
    if not ok:
        return {"valid": False, "reasons": reasons, "scores": {}}
    scores = oracle.score(product_ids)
    valid_scores = [v for v in scores.values() if not math.isnan(v)]
    return {
        "valid": True, "reasons": [], "scores": scores,
        "mean_pIC50": sum(valid_scores) / max(len(valid_scores), 1),
    }

Overwriting local_eval.py


## Nim Oracle

In [7]:
%%writefile nim_oracle.py
"""NVIDIA NIM Boltz-2 oracle with disk cache and rate limiting."""
import hashlib
import json
import math
import time
import urllib.error
import urllib.request
from pathlib import Path

ENDPOINT = "https://health.api.nvidia.com/v1/biology/mit/boltz2/predict"

class NimError(RuntimeError):
    pass

class NimOracle:
    def __init__(self, api_key, library, target_sequence,
                 cache_path="nim_cache.json", msa_path=None,
                 sampling_steps=200, diffusion_samples=5,
                 requests_per_minute=25, timeout=600): # <-- Changed to 25 RPM
        self.api_key = api_key
        self.library = library
        self.target_sequence = target_sequence
        self.cache_path = Path(cache_path)
        self.msa_path = msa_path
        self.sampling_steps = sampling_steps
        self.diffusion_samples = diffusion_samples
        self.min_interval = 60.0 / requests_per_minute
        self.timeout = timeout
        self._last_call = 0.0
        self._cache = self._load_cache()

    def _load_cache(self):
        return json.loads(self.cache_path.read_text()) if self.cache_path.exists() else {}

    def _save_cache(self):
        tmp = self.cache_path.with_suffix(".tmp")
        tmp.write_text(json.dumps(self._cache, indent=0))
        tmp.replace(self.cache_path)

    def _key(self, smiles):
        payload = f"{self.target_sequence}|{smiles}|{self.sampling_steps}|{self.diffusion_samples}"
        return hashlib.sha256(payload.encode()).hexdigest()[:20]

    def _wait(self):
        elapsed = time.monotonic() - self._last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_call = time.monotonic()

    def score(self, product_ids):
        results = {}
        for pid in product_ids:
            smiles = self.library.resolve(pid)
            if smiles is None:
                results[pid] = float("nan")
                continue
            key = self._key(smiles)
            if key in self._cache:
                results[pid] = self._cache[key]
                continue
            try:
                pIC50 = self._call_nim(smiles)
            except NimError as exc:
                print(f"  NIM error on {pid}: {exc}")
                pIC50 = float("nan")
            results[pid] = pIC50
            if not math.isnan(pIC50):
                self._cache[key] = pIC50
                self._save_cache()
        return results

    def _call_nim(self, smiles):
        self._wait()
        payload = {
            "polymers": [{"id": "A", "molecule_type": "protein",
                          "sequence": self.target_sequence}],
            "ligands": [{"id": "LIG", "smiles": smiles, "predict_affinity": True}],
            "sampling_steps_affinity": self.sampling_steps,
            "diffusion_samples_affinity": self.diffusion_samples,
        }
        if self.msa_path:
            payload["msas"] = [{"id": "A", "path": str(self.msa_path)}]

        req = urllib.request.Request(
            ENDPOINT, data=json.dumps(payload).encode(), method="POST",
            headers={"Authorization": f"Bearer {self.api_key}",
                     "Content-Type": "application/json"})
        
        for attempt in range(3):
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                    body = json.load(resp)
                    return float(body["affinities"]["LIG"]["affinity_pic50"][0])
            except urllib.error.HTTPError as exc:
                if exc.code == 429:
                    print(f"  Rate limited (429). Sleeping 10s before retry {attempt+1}/3...")
                    time.sleep(10)
                    continue
                raise NimError(f"HTTP {exc.code}: {exc.read(512).decode(errors='replace')}") from None
            except (urllib.error.URLError, OSError) as exc:
                raise NimError(f"request failed: {exc}") from None
            except (KeyError, IndexError, TypeError, ValueError):
                return float("nan")
                
        print("  Failed after 3 retries due to rate limiting.")
        return float("nan")

Overwriting nim_oracle.py


## Build a synthetic amide-coupling library

In [8]:
%%writefile make_synthetic_library.py
"""Dynamically build a synthetic amide-coupling library of any size."""
import hashlib
import json
import math
import sqlite3
import sys
from pathlib import Path

ROLE_A, ROLE_B, REACTION_ID = 1, 2, 1
SMARTS = "[C:1](=[O:2])[OH:3].[N:4]>>[C:1](=[O:2])[N:4]"

FILTERS = {
    "max_kappa1": 20, "max_molecular_weight": 600, "max_heavy_atoms": 45,
    "min_heavy_atoms": 5, "min_molecular_weight": 100,  # <-- Relaxed to 100
    "max_aromatic_rings": 4, "max_aliphatic_rings": 4, "min_rings": 0,
    "max_fsp3": 1.0, "max_spiro_atoms": 2, "max_ring_size": 8,
    "max_aromatic_nitrogen": 4, "max_aromatic_fraction": 1.0, "max_aromatic_carbon": 20,
    "max_halogens": 5, "max_nitrogen": 6, "max_oxygen": 8,
    "max_sulfur": 2, "max_phosphorus": 1, "max_iodine": 1,
    "max_bromine": 2, "max_fluorine": 5,
    "min_heteroatom_fraction": 0.0, "max_heteroatom_fraction": 1.0,
    "max_hbd": 5, "max_amide_bonds": 3, "max_nitrile": 2,
    "max_hydroxyl": 3, "max_triple_bonds": 2, "min_rotatable_bonds": 0,
    "max_rotatable_bonds": 12, "max_bertz_ct": 1000,
    "max_hba": 10, "max_kappa2": 20, "max_kappa3": 20,
    "max_stereocenters": 4, "max_logp": 6.0,
}

def _generate_smiles(idx, is_acid):
    cores = ["c1ccccc1", "c1ccncc1", "c1cncnc1", "C1CCCCC1", "C1CCNCC1", 
             "C1CCOCC1", "c1ccsc1", "c1ccoc1", "c1ccc2ccccc2c1", "c1ccc2[nH]ccc2c1"]
    subs = ["", "F", "Cl", "Br", "C", "CC", "OC", "N", "C#N", 
            "C(F)(F)F", "O", "S", "C(=O)O", "N(C)C", "S(=O)(=O)C"]
    
    core = cores[idx % len(cores)]
    sub = subs[(idx // len(cores)) % len(subs)]
    chain = "C" * ((idx // (len(cores) * len(subs))) % 10)
    
    if is_acid:
        if sub:
            if core == "c1ccccc1":
                smi = f"O=C([1*]){chain}c1ccc({sub})cc1" if chain else f"O=C([1*])c1ccc({sub})cc1"
            else:
                smi = f"{sub}{core}{chain}C(=O)[1*]"
        else:
            smi = f"{core}{chain}C(=O)[1*]"
    else:
        if sub:
            if core == "c1ccccc1":
                smi = f"[2*]N{chain}c1ccc({sub})cc1" if chain else f"[2*]Nc1ccc({sub})cc1"
            else:
                smi = f"[2*]N{sub}{core}{chain}"
        else:
            smi = f"[2*]N{core}{chain}" if chain else f"[2*]N{core}"
    
    return smi

def build(out_dir, target_count):
    # Use a 10x buffer to guarantee we have enough valid products
    block_count = math.ceil(math.sqrt(target_count * 10))
    
    ACIDS = [(_generate_smiles(i, True), 1000 + i) for i in range(block_count)]
    AMINES = [(_generate_smiles(i, False), 2000 + i) for i in range(block_count)]

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    db_path = out_dir / "molecules.sqlite"
    db_path.unlink(missing_ok=True)
    
    conn = sqlite3.connect(db_path)
    conn.executescript("""
        CREATE TABLE molecules (mol_id INTEGER PRIMARY KEY, smiles TEXT, role_mask INTEGER);
        CREATE TABLE reactions (rxn_id TEXT PRIMARY KEY, smarts TEXT,
                                roleA INTEGER, roleB INTEGER, roleC INTEGER);
    """)
    for smi, mid in ACIDS:
        conn.execute("INSERT INTO molecules VALUES (?, ?, ?)", (mid, smi, ROLE_A))
    for smi, mid in AMINES:
        conn.execute("INSERT INTO molecules VALUES (?, ?, ?)", (mid, smi, ROLE_B))
    conn.execute("INSERT INTO reactions VALUES (?, ?, ?, ?, ?)",
                 (str(REACTION_ID), SMARTS, ROLE_A, ROLE_B, 0))
    conn.commit()
    conn.close()
    
    manifest = {
        "format_version": 1,
        "rdkit_version": "2024.09.4",
        "allowed_reaction": f"rxn:{REACTION_ID}",
        "case_digest": hashlib.sha256(f"dynamic-{target_count}".encode()).hexdigest(),
        "database_sha256": hashlib.sha256(db_path.read_bytes()).hexdigest(),
        "filters": FILTERS,
    }
    (out_dir / "library.json").write_text(json.dumps(manifest, indent=2))
    print(f"Built {out_dir}: {len(ACIDS)} acids x {len(AMINES)} amines = {len(ACIDS)*len(AMINES):,} possible products")

if __name__ == "__main__":
    target = int(sys.argv[1]) if len(sys.argv) > 1 else 10000
    out = sys.argv[2] if len(sys.argv) > 2 else f"synthetic-library-{target}"
    build(out, target)

Overwriting make_synthetic_library.py


## Generate a random candidate pool from the owner's frozen chemistry library

In [9]:
%%writefile generate_pool.py
"""Generate a candidate pool from a frozen chemistry library (dynamic size)."""
import csv
import itertools
import random

from rdkit import Chem
from chemistry import Library, passes_filters

def generate(library, count, attempts, seed):
    _, role_a, role_b, role_c = library.reaction()
    pools = [library.ids_for_role(role) for role in (role_a, role_b, role_c) if role]
    if len(pools) not in (2, 3) or not all(pools):
        raise ValueError("The reaction has an empty or invalid building-block role")
    
    capacity = 1
    for p in pools:
        capacity *= len(p)
    
    rng = random.Random(seed)
    chemical_keys, accepted = set(), []
    
    # Force enumeration for anything up to 2M combos
    use_enumeration = capacity < 2_000_000
    
    if use_enumeration:
        print(f"  [Strategy: ENUMERATION] Pool capacity = {capacity:,}, count = {count:,}")
        all_combos = list(itertools.product(*pools))
        rng.shuffle(all_combos)
        combos = all_combos
    else:
        print(f"  [Strategy: RANDOM SAMPLING] {attempts:,} attempts, capacity = {capacity:,}")
        sampled = set()
        combos = []
        for _ in range(attempts):
            identifiers = tuple(rng.choice(pool) for pool in pools)
            if identifiers in sampled:
                continue
            sampled.add(identifiers)
            combos.append(identifiers)
    
    for identifiers in combos:
        variants = list(enumerate(library.products(identifiers)))
        rng.shuffle(variants)
        for index, smiles in variants:
            mol = Chem.MolFromSmiles(smiles)
            if mol is None or not passes_filters(mol, library.manifest["filters"]):
                continue
            key = Chem.MolToInchiKey(mol)
            if not key or key in chemical_keys:
                continue
            chemical_keys.add(key)
            identifier = "rxn:" + ":".join(map(str, (library.reaction_id, *identifiers)))
            if index:
                identifier += f":k{index}"
            accepted.append({"product_id": identifier, "smiles": smiles})
            if len(accepted) >= count:
                return accepted, len(combos)
    
    return accepted, len(combos)

def main(library_dir="synthetic-library", reaction="rxn:1", count=100, attempts=500, seed=42, output="pool.csv"):
    if count < 1 or attempts < 1:
        raise ValueError("count and attempts must be positive")
    try:
        with Library(library_dir) as library:
            if reaction != library.manifest["allowed_reaction"]:
                raise ValueError(f"reaction '{reaction}' must match '{library.manifest['allowed_reaction']}'")
            accepted, processed = generate(library, count, attempts, seed)
            print(f"  Library case digest: {library.manifest['case_digest']}")
    except (ValueError, OSError) as exc:
        raise RuntimeError(f"Error generating pool: {exc}")

    with open(output, "w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["product_id", "smiles"])
        writer.writeheader()
        writer.writerows(accepted)
    
    status = "✅" if len(accepted) >= count else "⚠️"
    print(f"  {status} Saved {len(accepted):,} / {count:,} requested; {processed:,} combos processed")
    if len(accepted) < count:
        print(f"  ⚠️  SHORTFALL: requested {count:,}, got {len(accepted):,}")
    
    return accepted

if __name__ == "__main__":
    main()

Overwriting generate_pool.py


In [10]:
# """Generate a random candidate pool from the owner's frozen chemistry library."""
# import csv
# import random

# from rdkit import Chem

# # from chemistry import Library, passes_filters


# def generate(library, count, attempts, seed):
#     _, role_a, role_b, role_c = library.reaction()
#     pools = [library.ids_for_role(role) for role in (role_a, role_b, role_c) if role]
#     if len(pools) not in (2, 3) or not all(pools):
#         raise ValueError("The reaction has an empty or invalid building-block role")
#     rng = random.Random(seed)
#     sampled, chemical_keys, accepted = set(), set(), []
#     for _ in range(attempts):
#         identifiers = tuple(rng.choice(pool) for pool in pools)
#         if identifiers in sampled:
#             continue
#         sampled.add(identifiers)
#         variants = list(enumerate(library.products(identifiers)))
#         rng.shuffle(variants)
#         for index, smiles in variants:
#             mol = Chem.MolFromSmiles(smiles)
#             if mol is None or not passes_filters(mol, library.manifest["filters"]):
#                 continue
#             key = Chem.MolToInchiKey(mol)
#             if not key or key in chemical_keys:
#                 continue
#             chemical_keys.add(key)
#             identifier = "rxn:" + ":".join(map(str, (library.reaction_id, *identifiers)))
#             if index:
#                 identifier += f":k{index}"
#             accepted.append({"product_id": identifier, "smiles": smiles})
#             if len(accepted) >= count:
#                 return accepted, len(sampled)
#     return accepted, len(sampled)


# def main(library_dir="synthetic-library", reaction="rxn:1", count=100, attempts=500, seed=42, output="pool.csv"):
#     """Clean entry point for notebooks and scripts."""
#     if count < 1 or attempts < 1:
#         raise ValueError("count and attempts must be positive")
        
#     try:
#         with Library(library_dir) as library:
#             if reaction != library.manifest["allowed_reaction"]:
#                 raise ValueError(f"reaction '{reaction}' must match the library's allowed reaction: '{library.manifest['allowed_reaction']}'")
            
#             accepted, attempted = generate(library, count, attempts, seed)
#             print(f"Library case digest: {library.manifest['case_digest']}")
            
#     except (ValueError, OSError) as exc:
#         raise RuntimeError(f"Error generating pool: {exc}")

#     # Save the output
#     with open(output, "w", newline="") as stream:
#         writer = csv.DictWriter(stream, fieldnames=["product_id", "smiles"])
#         writer.writeheader()
#         writer.writerows(accepted)
        
#     print(f"Saved {len(accepted)} candidates to {output}; {attempted} distinct combinations attempted.")
#     if len(accepted) < count:
#         print("The requested pool size was not reached; increase attempts or inspect the allowed library.")
#     print("Local filtering does not establish historical novelty or whole-set diversity; use the evaluation service.")
    
#     return accepted


# if __name__ == "__main__":
#     # If run from command line, it will use the defaults above.
#     main()

## Metrics

In [11]:
%%writefile metrics.py
"""Compare checkpoints from baseline.py, search.py, or search_nim.py.

Handles both schemas:
- baseline.py / search.py: state["probes"] = [{result:{items:[{product,score}], backend}}]
- search_nim.py: state["scored"] = [[product, score], ...]
"""
import os
import csv
import json
import sys
from pathlib import Path


def _normalize(state):
    """Return [(product_id, score)] regardless of source schema."""
    if state.get("scored"):
        return [(p, s) for p, s in state["scored"]]
    out = []
    for job in state.get("probes", []):
        res = job.get("result") or {}
        backend = res.get("backend", "reference")
        for item in res.get("items", []):
            pid = item.get("product")
            score = item.get("demo_value") if backend == "demo" else item.get("score")
            if pid and isinstance(score, (int, float)):
                out.append((pid, float(score)))
    return out


def _validation(state):
    v = state.get("validation") or {}
    return v.get("result") or v


def _final_scores(state):
    """Return final-set scores (list of floats) or [] if not run."""
    final = state.get("final") or {}
    # search_nim.py schema
    if "scores" in final:
        return [float(v) for v in final["scores"].values()
                if isinstance(v, (int, float)) and v == v]
    # baseline.py schema
    res = final.get("result") or {}
    backend = res.get("backend", "reference")
    out = []
    for item in res.get("items", []):
        v = item.get("demo_value") if backend == "demo" else item.get("score")
        if isinstance(v, (int, float)):
            out.append(float(v))
    return out


def compute(state, fps_by_pid=None):
    case = state.get("case", {}) or {}
    target_count = case.get("target_count", 1)
    required = case.get("required_count", 3)
    probes = _normalize(state)
    selected = state.get("selected") or []
    validation = _validation(state)
    final_scores = _final_scores(state)
    scores = [s for _, s in probes]

    # search_nim.py does not pay per-pair for final (cache hit), baseline does
    if state.get("scored") is not None:
        pairs_charged = len(probes)  # one NIM call per probe
    else:
        pairs_charged = (len(state.get("probes", [])) + (required if final_scores else 0)) * target_count

    metrics = {
        "checkpoint_strategy": state.get("strategy", "baseline"),
        "case": case.get("id", "synthetic-demo"),
        "seed": state.get("seed"),
        "n_probes": len(probes),
        "n_unique_scored": len(set(p for p, _ in probes)),
        "n_selected": len(selected),
        "required_count": required,
        "pairs_charged": pairs_charged,
        "validation_valid": validation.get("valid"),
        "final_valid": (state.get("final") or {}).get("valid"),
        "status": state.get("status", "completed"),
    }
    if scores:
        top = sorted(scores, reverse=True)
        metrics["best_probe_score"] = top[0]
        metrics["mean_top_k_probe"] = sum(top[:required]) / max(required, 1)
        metrics["mean_all_probe"] = sum(scores) / len(scores)
        metrics["best_per_pair"] = top[0] / pairs_charged if pairs_charged else 0.0
    if final_scores:
        metrics["best_final_score"] = max(final_scores)
        metrics["mean_final_score"] = sum(final_scores) / len(final_scores)

    if fps_by_pid and len(selected) >= 2:
        from rdkit import DataStructs
        fps = [fps_by_pid[p] for p in selected if p in fps_by_pid]
        if len(fps) >= 2:
            sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j])
                    for i in range(len(fps)) for j in range(i + 1, len(fps))]
            metrics["div_mean_sim"] = sum(sims) / len(sims)
            metrics["div_max_sim"] = max(sims)
    return metrics


def load_fps(pool_path):
    try:
        from rdkit import Chem
        from rdkit.Chem import AllChem
    except ImportError:
        print("WARN: rdkit not installed; diversity metrics skipped", file=sys.stderr)
        return None
    fps = {}
    with open(pool_path, newline="") as stream:
        for row in csv.DictReader(stream):
            pid = (row.get("product_id") or "").strip()
            smi = (row.get("smiles") or "").strip()
            if not pid or not smi:
                continue
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                fps[pid] = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
    return fps


def main(checkpoints, pool=None, csv_out=None):
    """Core logic with parametrized signatures."""
    fps = load_fps(pool) if pool else None
    rows = []
    for path in checkpoints:
        try:
            state = json.loads(Path(path).read_text())
        except (OSError, json.JSONDecodeError) as exc:
            print(f"WARN: cannot read {path}: {exc}", file=sys.stderr); continue
        row = compute(state, fps)
        row["checkpoint"] = Path(path).name
        rows.append(row)

    if not rows:
        print("No checkpoints read", file=sys.stderr); return 1

    columns = ["checkpoint", "case", "checkpoint_strategy", "seed", "n_probes",
               "n_unique_scored", "n_selected", "pairs_charged",
               "best_probe_score", "mean_top_k_probe", "best_per_pair",
               "best_final_score", "validation_valid", "final_valid",
               "div_mean_sim", "div_max_sim"]
    print("\t".join(columns))
    for row in rows:
        print("\t".join(str(row.get(c, "")) for c in columns))

    if csv_out:
        with open(csv_out, "w", newline="") as stream:
            w = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
            w.writeheader(); w.writerows(rows)
        print(f"\nSaved {csv_out}", file=sys.stderr)
    return 0

RESULTS_DIR = "/kaggle/working/results"
POOL_PATH = "pool.csv"
CSV_OUT = f"{RESULTS_DIR}/phase2-seeds.csv"

result_files = [
    os.path.join(RESULTS_DIR, f) 
    for f in os.listdir(RESULTS_DIR) 
    if f.endswith(".json") and f.startswith("search-")
]

# metrics_main(result_files, pool=POOL_PATH, csv_out=CSV_OUT)

Overwriting metrics.py


## Search Nim

In [12]:
%%writefile search_nim.py
import csv
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem
from scipy.stats import norm
from sklearn.ensemble import RandomForestRegressor

RDLogger.DisableLog('rdApp.*')
from chemistry import Library
from local_eval import validate_set, final_set
from nim_oracle import NimOracle

_MORGAN_GEN = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

def _load_pool(path):
    pids, smiles_map = [], {}
    with open(path, newline="") as stream:
        for row in csv.DictReader(stream):
            pid = row.get("product_id")
            smi = row.get("smiles")
            if pid and smi:
                pids.append(pid)
                smiles_map[pid] = smi
    return pids, smiles_map

def _maxmin(pids, k, fps_cache):
    if len(pids) <= k:
        return pids[:k]
    valid = [pid for pid in pids if pid in fps_cache]
    if len(valid) <= k:
        return valid
    chosen, remaining = [valid[0]], valid[1:]
    while len(chosen) < k and remaining:
        best, best_sim = None, 2.0
        for pid in remaining:
            sim = max(DataStructs.TanimotoSimilarity(fps_cache[pid], fps_cache[c]) for c in chosen)
            if sim < best_sim:
                best, best_sim = pid, sim
        chosen.append(best); remaining.remove(best)
    return chosen

def _select_batch(state, remaining, batch_size, fps_cache):
    strategy, scored = state["strategy"], state["scored"]
    if strategy == "random" or len(scored) < 2 * batch_size:
        rng = random.Random(state["seed"] + len(scored))
        sample_size = min(len(remaining), batch_size * 10)
        shortlist = rng.sample(remaining, sample_size)
        return _maxmin(shortlist, batch_size, fps_cache)

    pids = [p for p, _ in scored if p in fps_cache]
    y = np.array([s for p, s in scored if p in fps_cache], dtype=np.float64)
    
    X_train = np.zeros((len(pids), 2048), dtype=np.float32)
    for i, pid in enumerate(pids):
        DataStructs.ConvertToNumpyArray(fps_cache[pid], X_train[i])
        
    rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
                               random_state=state["seed"], n_jobs=-1).fit(X_train, y)
    
    X_rem = np.zeros((len(remaining), 2048), dtype=np.float32)
    valid_rem = []
    for i, pid in enumerate(remaining):
        if pid in fps_cache:
            DataStructs.ConvertToNumpyArray(fps_cache[pid], X_rem[i])
            valid_rem.append(pid)
            
    preds = np.array([t.predict(X_rem) for t in rf.estimators_])
    mu, sigma = preds.mean(axis=0), preds.std(axis=0) + 1e-9

    if strategy == "greedy":
        acq = mu
    elif strategy == "ucb":
        acq = mu + 2.0 * sigma
    else:  # ei
        best = y.max()
        z = (mu - best) / sigma
        acq = (mu - best) * norm.cdf(z) + sigma * norm.pdf(z)

    ranked = [valid_rem[i] for i in np.argsort(-acq)]
    shortlist = ranked[:max(50, batch_size * 10)]
    return _maxmin(shortlist, batch_size, fps_cache)

def _state_path(output, strategy, seed):
    return Path(str(output) + f".{strategy}.s{seed}.state")

def _save(path, state):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2))
    tmp.replace(path)

def run(library_dir, pool_path, target_sequence, strategy="ei", seed=42, probes=24,
        batch_size=4, required=3, max_sim=0.6, msa=None, cache="nim_cache.json",
        output="search-nim.json", resume=False):
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    state_path = _state_path(output, strategy, seed)

    if resume and not state_path.exists():
        raise SystemExit(f"--resume requested but {state_path} does not exist.")

    with Library(library_dir) as library:
        if state_path.exists():
            state = json.loads(state_path.read_text())
            print(f"Resuming {state_path}: {len(state['scored'])} scored")
        else:
            if output.exists():
                raise SystemExit(f"{output} exists; use --resume or a new --output")
            pids, smiles_map = _load_pool(pool_path)
            random.Random(seed).shuffle(pids)
            state = {
                "seed": seed, "strategy": strategy,
                "batch_size": batch_size, "probes_planned": probes,
                "max_sim": max_sim, "target_sequence": target_sequence,
                "pool": pids, "scored": [], "attempted": [],
                "smiles_map": smiles_map,
                "selected": None, "validation": None, "final": None,
                "history": [], "elapsed": 0.0,
            }
            _save(state_path, state)

        print("Precomputing fingerprints...")
        fps_cache = {}
        smiles_map = state.get("smiles_map", {})
        for pid in state["pool"]:
            smi = smiles_map.get(pid)
            if smi:
                mol = Chem.MolFromSmiles(smi)
                if mol:
                    fps_cache[pid] = _MORGAN_GEN.GetFingerprint(mol)
        print(f"Cached {len(fps_cache)} fingerprints.")

        oracle = NimOracle(
            api_key=os.environ["NVIDIA_API_KEY"], library=library,
            target_sequence=state["target_sequence"],
            cache_path=cache, msa_path=msa)

        attempted = set(state.get("attempted", []))
        remaining = [p for p in state["pool"] if p not in attempted]

        started = time.monotonic()
        previous = state.get("elapsed", 0.0)

        while len(state["scored"]) < state["probes_planned"] and remaining:
            batch = [p for p in _select_batch(state, remaining, state["batch_size"], fps_cache)
                     if p not in attempted]
            if not batch:
                break
            scores = oracle.score(batch)
            new = 0
            for pid in batch:
                attempted.add(pid)
                s = scores.get(pid, float("nan"))
                if not math.isnan(s):
                    state["scored"].append((pid, s)); new += 1
            state["attempted"] = sorted(attempted)
            remaining = [p for p in remaining if p not in attempted]
            if state["scored"]:
                state["history"].append(max(s for _, s in state["scored"]))
            state["elapsed"] = previous + time.monotonic() - started
            _save(state_path, state)
            best = max(s for _, s in state["scored"]) if state["scored"] else float("nan")
            print(f"  +{new} valid | {len(state['scored'])}/{state['probes_planned']} "
                  f"probes | best={best:.2f}")

        if state["selected"] is None:
            state["scored"].sort(key=lambda x: -x[1])
            scored_pids = [p for p, _ in state["scored"]]
            selected = []
            for pid in scored_pids:
                if pid not in fps_cache:
                    continue
                fp = fps_cache[pid]
                is_diverse = True
                for s in selected:
                    if DataStructs.TanimotoSimilarity(fp, fps_cache[s]) > max_sim:
                        is_diverse = False
                        break
                if is_diverse:
                    selected.append(pid)
                    if len(selected) == required:
                        break
            if len(selected) < required:
                for pid in scored_pids:
                    if pid not in selected:
                        selected.append(pid)
                        if len(selected) == required:
                            break
            state["selected"] = selected
            _save(state_path, state)

        ok, reasons, _ = validate_set(state["selected"], library, max_sim)
        state["validation"] = {"valid": ok, "reasons": reasons}
        _save(state_path, state)
        if not ok:
            print("Validation failed:", reasons)
            return 1

        state["final"] = final_set(state["selected"], library, oracle, max_sim)
        _save(state_path, state)
        output.write_text(json.dumps(state, indent=2))

        best = max(s for _, s in state["scored"]) if state["scored"] else float("nan")
        print(f"\nDone. best_pIC50={best:.2f} pairs={len(state['scored'])} "
              f"selected={len(state['selected'])} → {output}")
        return 0

def search_main(library_dir, pool_path, target_sequence, strategy="ei", seed=42, probes=24,
         batch_size=4, required=3, max_sim=0.6, msa=None, cache="nim_cache.json",
         output="search-nim.json", resume=False):
    return run(
        library_dir=library_dir, pool_path=pool_path, target_sequence=target_sequence,
        strategy=strategy, seed=seed, probes=probes, batch_size=batch_size,
        required=required, max_sim=max_sim, msa=msa, cache=cache,
        output=output, resume=resume
    )

Overwriting search_nim.py


In [13]:
# """Surrogate-guided search using NVIDIA NIM Boltz-2 as the oracle."""

# import csv
# import json
# import math
# import os
# import random
# import time
# from pathlib import Path

# import numpy as np
# from rdkit import Chem, DataStructs, RDLogger
# from rdkit.Chem import AllChem
# from scipy.stats import norm
# from sklearn.ensemble import RandomForestRegressor

# # Suppress RDKit warning spam
# RDLogger.DisableLog('rdApp.*')

# # from chemistry import Library
# # from local_eval import validate_set, final_set
# # from nim_oracle import NimOracle

# # Initialize the modern Morgan Generator once
# _MORGAN_GEN = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

# def _load_pool(path):
#     with open(path, newline="") as stream:
#         return [row["product_id"] for row in csv.DictReader(stream) if row.get("product_id")]

# def _fp_matrix(pids, library):
#     X = np.zeros((len(pids), 2048), dtype=np.float32)
#     for i, pid in enumerate(pids):
#         smi = library.resolve(pid)
#         if smi is None:
#             continue
#         mol = Chem.MolFromSmiles(smi)
#         if mol is None:
#             continue
#         fp = _MORGAN_GEN.GetFingerprint(mol)
#         DataStructs.ConvertToNumpyArray(fp, X[i])
#     return X

# def _maxmin(pids, k, library):
#     if len(pids) <= k:
#         return pids[:k]
#     fps, valid = {}, []
#     for pid in pids:
#         smi = library.resolve(pid)
#         if smi is None:
#             continue
#         mol = Chem.MolFromSmiles(smi)
#         if mol is None:
#             continue
#         fps[pid] = _MORGAN_GEN.GetFingerprint(mol)
#         valid.append(pid)
#     if len(valid) <= k:
#         return valid
#     chosen, remaining = [valid[0]], valid[1:]
#     while len(chosen) < k and remaining:
#         best, best_sim = None, 2.0
#         for pid in remaining:
#             sim = max(DataStructs.TanimotoSimilarity(fps[pid], fps[c]) for c in chosen)
#             if sim < best_sim:
#                 best, best_sim = pid, sim
#         chosen.append(best); remaining.remove(best)
#     return chosen

# def _select_batch(state, remaining, library, batch_size):
#     strategy, scored = state["strategy"], state["scored"]
    
#     if strategy == "random" or len(scored) < 2 * batch_size:
#         rng = random.Random(state["seed"] + len(scored))
#         sample_size = min(len(remaining), batch_size * 10)
#         shortlist = rng.sample(remaining, sample_size)
#         return _maxmin(shortlist, batch_size, library)

#     pids = [p for p, _ in scored]
#     y = np.array([s for _, s in scored], dtype=np.float64)
#     rf = RandomForestRegressor(n_estimators=100, min_samples_leaf=2,
#                                random_state=state["seed"], n_jobs=-1).fit(
#         _fp_matrix(pids, library), y)
#     X_rem = _fp_matrix(remaining, library)
#     preds = np.array([t.predict(X_rem) for t in rf.estimators_])
#     mu, sigma = preds.mean(axis=0), preds.std(axis=0) + 1e-9

#     if strategy == "greedy":
#         acq = mu
#     elif strategy == "ucb":
#         acq = mu + 2.0 * sigma
#     else:  # ei
#         best = y.max()
#         z = (mu - best) / sigma
#         acq = (mu - best) * norm.cdf(z) + sigma * norm.pdf(z)

#     ranked = [remaining[i] for i in np.argsort(-acq)]
#     shortlist = ranked[:max(50, batch_size * 10)]
#     return _maxmin(shortlist, batch_size, library)

# def _state_path(output, strategy, seed):
#     return Path(str(output) + f".{strategy}.s{seed}.state")

# def _save(path, state):
#     path = Path(path)
#     path.parent.mkdir(parents=True, exist_ok=True)
#     tmp = path.with_suffix(".tmp")
#     tmp.write_text(json.dumps(state, indent=2))
#     tmp.replace(path)

# def run(library_dir, pool_path, target_sequence, strategy="ei", seed=42, probes=24,
#         batch_size=4, required=3, max_sim=0.6, msa=None, cache="nim_cache.json",
#         output="search-nim.json", resume=False):
#     output = Path(output)
#     output.parent.mkdir(parents=True, exist_ok=True)
#     state_path = _state_path(output, strategy, seed)

#     if resume and not state_path.exists():
#         raise SystemExit(f"--resume requested but {state_path} does not exist.")

#     with Library(library_dir) as library:
#         if state_path.exists():
#             state = json.loads(state_path.read_text())
#             print(f"Resuming {state_path}: {len(state['scored'])} scored, "
#                   f"{len(state.get('attempted', []))} attempted")
#         else:
#             if output.exists():
#                 raise SystemExit(f"{output} exists; use --resume or a new --output")
#             pids = _load_pool(pool_path)
#             random.Random(seed).shuffle(pids)
#             state = {
#                 "seed": seed, "strategy": strategy,
#                 "batch_size": batch_size, "probes_planned": probes,
#                 "max_sim": max_sim, "target_sequence": target_sequence,
#                 "pool": pids, "scored": [], "attempted": [],
#                 "selected": None, "validation": None, "final": None,
#                 "history": [], "elapsed": 0.0,
#             }
#             _save(state_path, state)

#         oracle = NimOracle(
#             api_key=os.environ["NVIDIA_API_KEY"], library=library,
#             target_sequence=state["target_sequence"],
#             cache_path=cache, msa_path=msa)

#         attempted = set(state.get("attempted", []))
#         remaining = [p for p in state["pool"] if p not in attempted]

#         started = time.monotonic()
#         previous = state.get("elapsed", 0.0)

#         while len(state["scored"]) < state["probes_planned"] and remaining:
#             batch = [p for p in _select_batch(state, remaining, library, state["batch_size"])
#                      if p not in attempted]
#             if not batch:
#                 break
#             scores = oracle.score(batch)
#             new = 0
#             for pid in batch:
#                 attempted.add(pid)
#                 s = scores.get(pid, float("nan"))
#                 if not math.isnan(s):
#                     state["scored"].append((pid, s)); new += 1
#             state["attempted"] = sorted(attempted)
#             remaining = [p for p in remaining if p not in attempted]
#             if state["scored"]:
#                 state["history"].append(max(s for _, s in state["scored"]))
#             state["elapsed"] = previous + time.monotonic() - started
#             _save(state_path, state)
#             best = max(s for _, s in state["scored"]) if state["scored"] else float("nan")
#             print(f"  +{new} valid | {len(state['scored'])}/{state['probes_planned']} "
#                   f"probes | best={best:.2f}")

#         if state["selected"] is None:
#             state["scored"].sort(key=lambda x: -x[1])
#             scored_pids = [p for p, _ in state["scored"]]
            
#             # STRICT DIVERSITY SELECTION
#             selected = []
#             fps_cache = {}
#             for pid in scored_pids:
#                 if pid not in fps_cache:
#                     smi = library.resolve(pid)
#                     if smi:
#                         mol = Chem.MolFromSmiles(smi)
#                         if mol:
#                             fps_cache[pid] = _MORGAN_GEN.GetFingerprint(mol)
                
#                 if pid not in fps_cache:
#                     continue
                    
#                 fp = fps_cache[pid]
#                 is_diverse = True
#                 for s in selected:
#                     if DataStructs.TanimotoSimilarity(fp, fps_cache[s]) > max_sim:
#                         is_diverse = False
#                         break
                
#                 if is_diverse:
#                     selected.append(pid)
#                     if len(selected) == required:
#                         break
            
#             if len(selected) < required:
#                 print(f"WARNING: Could not find {required} diverse molecules in the top probes. Found {len(selected)}.")
#                 for pid in scored_pids:
#                     if pid not in selected:
#                         selected.append(pid)
#                         if len(selected) == required:
#                             break
            
#             state["selected"] = selected
#             _save(state_path, state)

#         ok, reasons, _ = validate_set(state["selected"], library, max_sim)
#         state["validation"] = {"valid": ok, "reasons": reasons}
#         _save(state_path, state)
#         if not ok:
#             print("Validation failed:", reasons)
#             return 1

#         state["final"] = final_set(state["selected"], library, oracle, max_sim)
#         _save(state_path, state)
#         output.write_text(json.dumps(state, indent=2))

#         best = max(s for _, s in state["scored"]) if state["scored"] else float("nan")
#         print(f"\nDone. best_pIC50={best:.2f} pairs={len(state['scored'])} "
#               f"selected={len(state['selected'])} → {output}")
#         return 0

# def search_main(library_dir, pool_path, target_sequence, strategy="ei", seed=42, probes=24,
#          batch_size=4, required=3, max_sim=0.6, msa=None, cache="nim_cache.json",
#          output="search-nim.json", resume=False):
#     return run(
#         library_dir=library_dir, pool_path=pool_path, target_sequence=target_sequence,
#         strategy=strategy, seed=seed, probes=probes, batch_size=batch_size,
#         required=required, max_sim=max_sim, msa=msa, cache=cache,
#         output=output, resume=resume
#     )

In [14]:
# import os
# from kaggle_secrets import UserSecretsClient

# # Load API key
# user_secrets = UserSecretsClient()
# os.environ["NVIDIA_API_KEY"] = user_secrets.get_secret("NVIDIA_API_KEY")

# HIV_SEQ = (
#     "PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKMIGGIGGFIKVRQYD"
#     "QILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNF"
# )

# # Call main directly
# # from search_nim import main

# search_main(
#     library_dir="synthetic-library",
#     pool_path="pool.csv",
#     target_sequence=HIV_SEQ,
#     strategy="ei",
#     seed=42,
#     probes=24,
#     batch_size=4,
#     required=3,
#     max_sim=0.6,
#     output="/kaggle/working/results/search-ei-s42.json"
# )

## Orchestrator

In [15]:
%%writefile run_experiment.py
"""Dynamically run the FULL Phase 1-3 benchmark for ANY molecule count."""
import os
import shutil
import argparse
import subprocess
from pathlib import Path
import pandas as pd

# Import local modules
from generate_pool import main as gen_main
from search_nim import search_main
from metrics import main as metrics_main

HIV_SEQ = ("PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKMIGGIGGFIKVRQYD"
           "QILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNF")

def main():
    parser = argparse.ArgumentParser(description="Run dynamic molecular search benchmark.")
    parser.add_argument("--count", type=int, required=True, help="Target pool size")
    parser.add_argument("--probes", type=int, default=200, help="NIM probes per strategy")
    parser.add_argument("--batch-size", type=int, default=10, help="Batch size")
    parser.add_argument("--required", type=int, default=5, help="Final set size required")
    parser.add_argument("--seeds", type=str, default="42", help="Comma-separated seeds (e.g., 42,43,44)")
    parser.add_argument("--ablation", action="store_true", help="If set, run greedy and ucb ablation")
    args = parser.parse_args()

    count = args.count
    seeds = [int(s.strip()) for s in args.seeds.split(",")]
    
    print(f"\n{'='*60}")
    print(f"🚀 Benchmarking {count:,} molecules | Seeds: {seeds} | Ablation: {args.ablation}")
    print(f"{'='*60}")

    # 1. Setup paths
    lib_dir = f"synthetic-library-{count}"
    out_dir = Path(f"/kaggle/working/results-{count}")
    out_dir.mkdir(parents=True, exist_ok=True)
    pool_csv = str(out_dir / f"pool-{count}.csv")
    cache_file = "nim_cache.json"

    # 2. Build Library dynamically
    print(f"\n[1/5] Building library {lib_dir}...")
    subprocess.run(["python", "make_synthetic_library.py", str(count), lib_dir], check=True)

    # 3. Generate Pool
    print(f"\n[2/5] Generating {count:,} candidate pool...")
    gen_main(
        library_dir=lib_dir, reaction="rxn:1", count=count,
        attempts=int(count * 3), seed=42, output=pool_csv
    )

    # 4. Run Strategies
    result_files = []
    
    # Phase 2 Ablation (Seed 42 only) - Greedy & UCB
    if args.ablation:
        for strategy in ["greedy", "ucb"]:
            print(f"\n[3/5] Ablation: {strategy.upper()} (Seed 42)...")
            out_file = str(out_dir / f"{strategy}-s42.json")
            search_main(
                library_dir=lib_dir, pool_path=pool_csv, target_sequence=HIV_SEQ,
                strategy=strategy, seed=42, probes=args.probes, batch_size=args.batch_size,
                required=args.required, max_sim=0.6, cache=cache_file, output=out_file
            )
            result_files.append(out_file)

    # Phase 1 & 2: Random vs EI across all seeds
    for seed in seeds:
        for strategy in ["random", "ei"]:
            print(f"\n[3/5] Running {strategy.upper()} (Seed {seed})...")
            out_file = str(out_dir / f"{strategy}-s{seed}.json")
            search_main(
                library_dir=lib_dir, pool_path=pool_csv, target_sequence=HIV_SEQ,
                strategy=strategy, seed=seed, probes=args.probes, batch_size=args.batch_size,
                required=args.required, max_sim=0.6, cache=cache_file, output=out_file
            )
            result_files.append(out_file)

    # 5. Compile Metrics (Phase 3)
    print(f"\n[4/5] Compiling metrics...")
    csv_out = str(out_dir / "metrics.csv")
    metrics_main(result_files, pool=pool_csv, csv_out=csv_out)

    print(f"\n[5/5] Phase 2 Summary")
    df = pd.read_csv(csv_out)
    summary = df.groupby("checkpoint_strategy").agg({
        "best_per_pair": ["mean", "std", "count"],
        "best_probe_score": ["mean"],
        "final_valid": lambda x: (x == True).sum()
    }).round(3)
    print(summary)
    
    summary.to_csv(out_dir / "summary.csv")
    shutil.make_archive(f"/kaggle/working/results-{count}", 'zip', out_dir)
    print(f"\n✅ Done! Zipped to /kaggle/working/results-{count}.zip")

if __name__ == "__main__":
    main()

Overwriting run_experiment.py


In [16]:
# !python run_experiment.py --count 1000 --probes 20 --seeds 42

In [17]:
!rm -rf /kaggle/working/results-100000
!rm -rf /kaggle/working/synthetic-library-100000
!python run_experiment.py --count 100000 --probes 200 --seeds 42


🚀 Benchmarking 100,000 molecules | Seeds: [42] | Ablation: False

[1/5] Building library synthetic-library-100000...
Built synthetic-library-100000: 1000 acids x 1000 amines = 1,000,000 possible products

[2/5] Generating 100,000 candidate pool...
  [Strategy: ENUMERATION] Pool capacity = 1,000,000, count = 100,000
  Library case digest: 804fc20aa552d1cd1e0b8cd2d6da1c76e1e3102f1024464c1225bdec1ec0f710
  ✅ Saved 100,000 / 100,000 requested; 1,000,000 combos processed

[3/5] Running RANDOM (Seed 42)...
Precomputing fingerprints...
Cached 100000 fingerprints.
  Rate limited (429). Sleeping 10s before retry 1/3...
  Rate limited (429). Sleeping 10s before retry 1/3...
  NIM error on rxn:1:1925:2047: HTTP 422: {"error":"Error processing request: No GPU available within 120.0s timeout"}
  +9 valid | 9/200 probes | best=7.24
  Rate limited (429). Sleeping 10s before retry 1/3...
  +10 valid | 19/200 probes | best=7.25
  Rate limited (429). Sleeping 10s before retry 1/3...
  Rate limited (429

## Phase 2 run: Ablation

In [18]:
# """Phase 2 Execution: Ablation and Seed Sweep for NIM Boltz-2."""
# import os
# from pathlib import Path
# # from search_nim import main as search_main

# LIBRARY_DIR = "synthetic-library"
# POOL_PATH = "pool.csv"
# OUTPUT_DIR = "/kaggle/working/results"
# HIV_SEQ = ("PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMSLPGRWKPKMIGGIGGFIKVRQYD"
#            "QILIEICGHKAIGTVLVGPTPVNIIGRNLLTQIGCTLNF")

# Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# def run_campaign(strategy: str, seed: int):
#     output_file = f"{OUTPUT_DIR}/search-{strategy}-s{seed}.json"
#     print(f"\n{'='*50}\nStrategy={strategy.upper()}, Seed={seed}\n{'='*50}")
#     try:
#         search_main(
#             library_dir=LIBRARY_DIR, pool_path=POOL_PATH, target_sequence=HIV_SEQ,
#             strategy=strategy, seed=seed, probes=24, batch_size=4, required=3,
#             max_sim=0.6, output=output_file
#         )
#     except Exception as e:
#         print(f"ERROR running {strategy} seed {seed}: {e}")

# def main():
#     print("--- Phase 2: Ablation (Seed 42) ---")
#     for strategy in ["random", "greedy", "ucb"]:
#         run_campaign(strategy, seed=42)
        
#     print("\n--- Phase 2: Seed Sweep (Seeds 43-46) ---")
#     for strategy in ["random", "ei"]:
#         for seed in [43, 44, 45, 46]:
#             run_campaign(strategy, seed=seed)

# if __name__ == "__main__":
#     main()

## Phase 2 compile

In [19]:
# """Compile Phase 2 Results: Generate metrics CSV and zip for download."""
# import os
# import shutil
# import pandas as pd
# # from metrics import main as metrics_main

# RESULTS_DIR = "/kaggle/working/results"
# POOL_PATH = "pool.csv"
# CSV_OUT = f"{RESULTS_DIR}/phase2-seeds.csv"
# ZIP_OUT = "/kaggle/working/phase_results"

# def main():
#     print("--- Compiling Metrics ---")
#     result_files = [
#         os.path.join(RESULTS_DIR, f) 
#         for f in os.listdir(RESULTS_DIR) 
#         if f.endswith(".json") and f.startswith("search-")
#     ]
    
#     if not result_files:
#         print("No results found! Did you run run_phase2.py?")
#         return

#     metrics_main(result_files, pool=POOL_PATH, csv_out=CSV_OUT)

#     print("\n" + "="*50 + "\nPHASE 2 SUMMARY (Mean best_per_pair)\n" + "="*50)
#     df = pd.read_csv(CSV_OUT)
    
#     summary = df.groupby("checkpoint_strategy").agg({
#         "best_per_pair": ["mean", "std", "count"],
#         "best_probe_score": ["mean"],
#         "final_valid": lambda x: (x == True).sum()
#     }).round(3)
    
#     print(summary)
    
#     print("\n--- Zipping Results ---")
#     shutil.make_archive(ZIP_OUT, 'zip', RESULTS_DIR)
#     print(f"Saved: {ZIP_OUT}.zip (Download this from the Kaggle 'Output' panel!)")

# if __name__ == "__main__":
#     main()